In [12]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [13]:
X_train = pd.read_parquet('X_train_cleaned.parquet')
y_train = pd.read_parquet('y_train.parquet')

In [14]:
cat_yes_no_cols = ['gender', 
                    'Partner', 
                    'Dependents', 
                    'PhoneService', 
                    'MultipleLines', 
                    'OnlineSecurity', 
                    'OnlineBackup', 
                    'DeviceProtection', 
                    'TechSupport', 
                    'StreamingTV', 
                    'StreamingMovies',
                    'PaperlessBilling']
cat_internetservice_cols = ['InternetService']
cat_contract_cols = ['Contract']
cat_paymentmethod_cols = ['PaymentMethod']

cat_yes_no_val_order = [['No', 'Yes']]*len(cat_yes_no_cols)
cat_internetservice_val_order = [['No', 'DSL', 'Fiber optic']]*len(cat_internetservice_cols)
cat_contract_val_order = [['Month-to-month', 'One year', 'Two year']]*len(cat_contract_cols)
cat_paymentmethod_val_order = [['Electronic check', 'Mailed check', 'Bank transfer (automatic)',
 'Credit card (automatic)']]*len(cat_paymentmethod_cols)


num_scale_cols = ['tenure', 'MonthlyCharges']
num_transform_cols = ['TotalCharges']

In [15]:
yesno_encoder_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot_yesno_encode', OneHotEncoder(categories=cat_yes_no_val_order, drop='first', sparse_output=False, handle_unknown='ignore')),
])

internetservice_encoder_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot_internetservice_encode', OneHotEncoder(categories=cat_internetservice_val_order, drop='first', sparse_output=False, handle_unknown='ignore')),
])

paymentmethod_encoder_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot_paymentmethod_encode', OneHotEncoder(categories=cat_paymentmethod_val_order, drop='first', sparse_output=False, handle_unknown='ignore')),
])

contract_encoder_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('ordinal_contract_encode', OrdinalEncoder(categories=cat_contract_val_order))
])

In [16]:
scalling_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=-1, add_indicator=True)),
    ('scalling', StandardScaler())
])

transform_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=-1, add_indicator=True)),
    ('transform', PowerTransformer())
])

In [17]:
linear_preprocessor = ColumnTransformer([
    ('ohe_yesno_encoder', yesno_encoder_pipeline, cat_yes_no_cols),
    ('ohe_internetservice_encoder', internetservice_encoder_pipeline, cat_internetservice_cols),
    ('ohe_paymentmethod_encoder', paymentmethod_encoder_pipeline, cat_paymentmethod_cols),
    ('ordinal_contract_encoder', contract_encoder_pipeline, cat_contract_cols),
    ('scalling', scalling_pipeline, num_scale_cols),
    ('transforming', transform_pipeline, num_transform_cols)
], remainder='passthrough')

tree_preprocessor = ColumnTransformer([
    ('ohe_yesno_encoder', yesno_encoder_pipeline, cat_yes_no_cols),
    ('ohe_internetservice_encoder', internetservice_encoder_pipeline, cat_internetservice_cols),
    ('ohe_paymentmethod_encoder', paymentmethod_encoder_pipeline, cat_paymentmethod_cols),
    ('ordinal_contract_encoder', contract_encoder_pipeline, cat_contract_cols)
], remainder='passthrough')

In [18]:
models = {
    'Logistic Regression': Pipeline([
        ('preprocessing', linear_preprocessor), 
        ('model', LogisticRegression(class_weight='balanced'))]),
    'KNN': Pipeline([
        ('preprocessing', linear_preprocessor), 
        ('model', KNeighborsClassifier())]),
    'Random Forest': Pipeline([
        ('preprocessing', tree_preprocessor), 
        ('model', RandomForestClassifier(class_weight='balanced'))]),
    'XGBoost': Pipeline([
        ('preprocessing', tree_preprocessor), 
        ('model', XGBClassifier())]),
    'LightGBM': Pipeline([
        ('preprocessing', tree_preprocessor), 
        ('model', LGBMClassifier(class_weight ='balanced'))])
}

In [19]:
results = {}

for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1_macro')

    results[name] = {
        'mean' : scores.mean().round(3),
        'std'  : scores.std().round(3)
    }


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all

[LightGBM] [Info] Number of positive: 1196, number of negative: 3311
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000990 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 620
[LightGBM] [Info] Number of data points in the train set: 4507, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Us

[LightGBM] [Info] Number of positive: 1196, number of negative: 3311
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000648 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 620
[LightGBM] [Info] Number of data points in the train set: 4507, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Lenovo\AppData\Local\Pa

[LightGBM] [Info] Number of positive: 1196, number of negative: 3311
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000445 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 620
[LightGBM] [Info] Number of data points in the train set: 4507, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Lenovo\AppData\Local\Pa

[LightGBM] [Info] Number of positive: 1196, number of negative: 3311
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001008 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 620
[LightGBM] [Info] Number of data points in the train set: 4507, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Lenovo\AppData\Local\Pa

[LightGBM] [Info] Number of positive: 1196, number of negative: 3312
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000573 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 620
[LightGBM] [Info] Number of data points in the train set: 4508, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


In [20]:
for model, res in results.items():
    print(f"Model  : {model}")
    print(f"Mean F1: {res['mean']}")
    print(f"Std F1 : {res['std']}")
    print("-" * 30)

Model  : Logistic Regression
Mean F1: 0.727
Std F1 : 0.019
------------------------------
Model  : KNN
Mean F1: 0.705
Std F1 : 0.019
------------------------------
Model  : Random Forest
Mean F1: 0.699
Std F1 : 0.014
------------------------------
Model  : XGBoost
Mean F1: 0.705
Std F1 : 0.015
------------------------------
Model  : LightGBM
Mean F1: 0.723
Std F1 : 0.011
------------------------------


In [21]:
pd.DataFrame(results).T.sort_values(by=['mean'], ascending=False)

,mean,std
Logistic Regression,0.727,0.019
LightGBM,0.723,0.011
KNN,0.705,0.019
XGBoost,0.705,0.015
Random Forest,0.699,0.014
